# Noised Images Construction

In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

In [2]:
DATASET_ROOT = Path("/home/rishi_677/Plant_Disease_DAC_Project/dataset")
SRC_DIR = DATASET_ROOT / 'raw_split' / 'test'
NOISED_ROOT = DATASET_ROOT / 'noised_test'

IMG_EXTS = {'.jpg'}

In [3]:
def salt_and_pepper(img_array, prob):
    noisy = img_array.copy()
    rng = np.random.default_rng()
    mask = rng.random(img_array.shape[:2])
    noisy[mask < (prob / 2)] = 255
    noisy[(mask >= prob / 2) & (mask < prob)] = 0
    return noisy.astype(np.uint8)


def gaussian_noise(img_array, mean=0.0, sigma=25.0):
    rng = np.random.default_rng()
    noise = rng.normal(mean, sigma, img_array.shape)
    noisy = np.clip(img_array.astype(np.float32) + noise, 0, 255)
    return noisy.astype(np.uint8)


def masking_noise(img_array, num_patches=4, patch_area_frac=0.05):
    noisy = img_array.copy()
    h, w = img_array.shape[:2]
    total = h * w

    for _ in range(num_patches):
        area = int(total * patch_area_frac)
        aspect = random.uniform(0.5, 2.0)
        ph = max(1, int((area / aspect) ** 0.5))
        pw = max(1, int(ph * aspect))
        ph, pw = min(ph, h), min(pw, w)
        top = random.randint(0, h - ph)
        left = random.randint(0, w - pw)
        noisy[top : top+ph, left : left+pw] = 0

    return noisy.astype(np.uint8)

In [4]:
VARIANTS = [{ 'name' : 'salt_pepper_0.03', 'label' : 'Salt & Pepper  (p=0.03)', 'fn' : salt_and_pepper, 'kwargs' : {'prob' : 0.03} },
    { 'name' : 'salt_pepper_0.05', 'label' : 'Salt & Pepper  (p=0.05)', 'fn' : salt_and_pepper, 'kwargs' : {'prob' : 0.05} },
    { 'name' : 'salt_pepper_0.07', 'label' : 'Salt & Pepper  (p=0.07)', 'fn' : salt_and_pepper, 'kwargs' : {'prob' : 0.07} },
    { 'name' : 'gaussian', 'label' : 'Gaussian  (Sigma = 25)', 'fn' : gaussian_noise, 'kwargs' : {'mean' : 0.0, 'sigma' : 25.0} },
    { 'name' : 'masking', 'label' : 'Masking  (4 patches)', 'fn' : masking_noise, 'kwargs' : {'num_patches' : 4, 'patch_area_frac' : 0.02} }]
    

print(f"{len(VARIANTS)} variants registered :\n")
for v in VARIANTS:
    print(f"{v['label']}")

5 variants registered :

Salt & Pepper  (p=0.03)
Salt & Pepper  (p=0.05)
Salt & Pepper  (p=0.07)
Gaussian  (Sigma = 25)
Masking  (4 patches)


In [5]:
out_dirs = {}

for v in VARIANTS:
    d = NOISED_ROOT / v['name']
    d.mkdir(parents=True, exist_ok=True)
    out_dirs[v['name']] = d
    print(f"Created : {d}")

print("\n All folders ready")

Created : /home/rishi_677/Plant_Disease_DAC_Project/dataset/noised_test/salt_pepper_0.03
Created : /home/rishi_677/Plant_Disease_DAC_Project/dataset/noised_test/salt_pepper_0.05
Created : /home/rishi_677/Plant_Disease_DAC_Project/dataset/noised_test/salt_pepper_0.07
Created : /home/rishi_677/Plant_Disease_DAC_Project/dataset/noised_test/gaussian
Created : /home/rishi_677/Plant_Disease_DAC_Project/dataset/noised_test/masking

 All folders ready


In [6]:
image_files = sorted(p for cls_dir in SRC_DIR.iterdir()  if cls_dir.is_dir() for p in cls_dir.iterdir()  if p.is_file() and p.suffix.lower() in IMG_EXTS)

for idx, img_path in enumerate(image_files, 1):
    cls_name = img_path.parent.name
    arr = np.array(Image.open(img_path).convert('RGB'))

    for v in VARIANTS:
        cls_out = out_dirs[v['name']] / cls_name
        cls_out.mkdir(parents=True, exist_ok=True)

        noisy = v['fn'](arr, **v['kwargs'])
        out_path = cls_out / (img_path.stem + '.JPG')
        Image.fromarray(noisy).save(out_path)

print("\n All noised images saved")


 All noised images saved
